# Day 24: Patch Embeddings — From Images to Sequences

> **Goal:** Learn how to convert images into a sequence of patch embeddings, making them compatible with the Transformer architecture.

---

## The Key Insight

Transformers process **sequences** of tokens. For NLP, tokens are words/subwords. For images, we need to create "visual tokens."

**Solution:** Divide the image into non-overlapping patches and project each patch into an embedding vector.

```
Image (3, 32, 32)  →  Split into 4×4 patches  →  64 patches of size (3, 4, 4)
                   →  Flatten each: (48,)
                   →  Linear projection: (48,) → (d_model,)
                   →  Sequence of 64 tokens, each of dimension d_model
```

### Optics Analogy
Think of patch embedding as **spatial sampling** — like a detector array where each pixel/superpixel captures local information, and the Transformer learns to integrate them globally.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Splitting Images into Patches (Manual)

Let's start by manually splitting an image into non-overlapping patches:

In [ ]:
# Load a sample CIFAR-10 image
transform = transforms.ToTensor()
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
img, label = dataset[0]
classes = dataset.classes

print(f"Image shape: {img.shape}")
print(f"Class: {classes[label]}")

# Visualize the original image
plt.figure(figsize=(3, 3))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Original: {classes[label]}")
plt.axis("off"); plt.show()

### Splitting into Patches

Now we'll manually split the image into a grid of `4×4` patches. A 32×32 image with patch size 4 gives us `8×8 = 64` patches. Each patch is a small `3×4×4` tensor (3 color channels, 4 pixels high, 4 pixels wide).

The visualization below shows all 64 patches arranged in their original spatial layout — you can see how the image is decomposed into small tiles.

In [ ]:
# Manual patch splitting
patch_size = 4
C, H, W = img.shape
num_patches_h = H // patch_size
num_patches_w = W // patch_size
num_patches = num_patches_h * num_patches_w

print(f"Image: {C}×{H}×{W}")
print(f"Patch size: {patch_size}×{patch_size}")
print(f"Grid: {num_patches_h}×{num_patches_w} = {num_patches} patches")

# Method 1: unfold
patches = img.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
print(f"\nAfter unfold: {patches.shape}")
# Reshape: (C, grid_h, grid_w, patch_h, patch_w) → (num_patches, C*patch_h*patch_w)
patches = patches.contiguous().view(C, -1, patch_size, patch_size)
patches = patches.permute(1, 0, 2, 3)  # (num_patches, C, patch_h, patch_w)
print(f"Patches: {patches.shape}")

# Visualize patches
fig, axes = plt.subplots(num_patches_h, num_patches_w, figsize=(8, 8))
for i in range(num_patches_h):
    for j in range(num_patches_w):
        idx = i * num_patches_w + j
        axes[i, j].imshow(patches[idx].permute(1, 2, 0))
        axes[i, j].axis("off")
        axes[i, j].set_title(f"P{idx}", fontsize=7)
plt.suptitle(f"Image split into {num_patches} patches ({patch_size}×{patch_size})", fontsize=14)
plt.tight_layout(); plt.show()

---
## 2. Linear Patch Projection

Each patch (3×4×4 = 48 values) gets projected to a `d_model`-dimensional embedding:

In [ ]:
# Linear projection of patches
d_model = 128
patch_dim = C * patch_size * patch_size  # 3 * 4 * 4 = 48

# Flatten patches and project
patches_flat = patches.reshape(num_patches, -1)  # (64, 48)
linear_proj = nn.Linear(patch_dim, d_model)

with torch.no_grad():
    embeddings = linear_proj(patches_flat)  # (64, 128)

print(f"Flattened patches: {patches_flat.shape}")
print(f"Patch embeddings:  {embeddings.shape}")
print(f"\nProjection params: {patch_dim} × {d_model} + {d_model} = {patch_dim * d_model + d_model:,}")

---
## 3. The Conv2d Trick — Elegant Patch Embedding

Instead of manual unfolding + linear projection, use a **single Conv2d** with `kernel_size=stride=patch_size`:

```python
nn.Conv2d(in_channels=3, out_channels=d_model, kernel_size=patch_size, stride=patch_size)
```

This is mathematically identical but much more efficient!

In [ ]:
# Conv2d trick for patch embedding
patch_conv = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)

with torch.no_grad():
    # img shape: (C, H, W), need batch dim → (1, C, H, W)
    conv_out = patch_conv(img.unsqueeze(0))  # (1, d_model, grid_h, grid_w)
    print(f"Conv output: {conv_out.shape}")

    # Flatten spatial dims and transpose
    tokens = conv_out.flatten(2).transpose(1, 2)  # (1, num_patches, d_model)
    print(f"Token sequence: {tokens.shape}")

# Verify: Conv2d trick gives same number of patches
print(f"\n✅ {tokens.shape[1]} tokens = {num_patches} patches")

---
## 4. The [CLS] Token

The **[CLS] (classification) token** is a special learnable token prepended to the patch sequence:

```
Sequence: [CLS] [Patch_1] [Patch_2] ... [Patch_N]
```

After passing through the Transformer, the [CLS] token's output is used for classification. It learns to **aggregate information** from all patches via attention.

In [ ]:
# Add [CLS] token
cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
print(f"CLS token shape: {cls_token.shape}")

# Prepend to sequence
batch_size = 4
tokens = torch.randn(batch_size, num_patches, d_model)
cls_expanded = cls_token.expand(batch_size, -1, -1)  # (B, 1, d_model)
sequence = torch.cat([cls_expanded, tokens], dim=1)   # (B, N+1, d_model)

print(f"\nPatch tokens:    {tokens.shape}")
print(f"CLS expanded:    {cls_expanded.shape}")
print(f"Full sequence:   {sequence.shape}  ← [CLS] + {num_patches} patches")
print(f"\nCLS is position 0: sequence[:, 0] will be used for classification")

---
## 5. Positional Encoding

Transformers have **no built-in sense of order**. Unlike CNNs (which use local spatial structure) or RNNs (which process sequentially), Transformers see all tokens simultaneously.

We must inject position information via **positional encoding**.

### Two main approaches:
| Type | Method | Used by |
|---|---|---|
| **Learned** | `nn.Parameter(randn(1, N+1, d_model))` | ViT (default) |
| **Sinusoidal** | Fixed sin/cos functions | Original Transformer |

In [ ]:
# Learned positional encoding
num_tokens = num_patches + 1  # patches + CLS
pos_embed = nn.Parameter(torch.randn(1, num_tokens, d_model) * 0.02)

print(f"Positional encoding shape: {pos_embed.shape}")
print(f"  Position 0 = CLS token position")
print(f"  Positions 1-{num_patches} = patch positions")

# Add to sequence (element-wise addition)
sequence_with_pos = sequence + pos_embed
print(f"\nSequence + position: {sequence_with_pos.shape}")

### Visualizing Sinusoidal Positional Encoding

While ViTs typically use **learned** positional embeddings, the original Transformer paper used **sinusoidal** (fixed) encoding. Below we visualize both the encoding patterns and the cosine similarity between positions — nearby positions should have higher similarity, creating a smooth spatial structure.

In [ ]:
# Visualize sinusoidal positional encoding (for understanding)
def sinusoidal_pos_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = sinusoidal_pos_encoding(65, 128)  # 65 positions (CLS + 64 patches)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = ax1.imshow(pe.numpy(), cmap='RdBu', aspect='auto')
ax1.set_xlabel("Embedding dimension"); ax1.set_ylabel("Position")
ax1.set_title("Sinusoidal Positional Encoding")
plt.colorbar(im, ax=ax1)

# Similarity between positions
sim = F.cosine_similarity(pe.unsqueeze(0), pe.unsqueeze(1), dim=-1)
im2 = ax2.imshow(sim.numpy(), cmap='viridis')
ax2.set_xlabel("Position j"); ax2.set_ylabel("Position i")
ax2.set_title("Cosine Similarity Between Positions")
plt.colorbar(im2, ax=ax2)

plt.tight_layout(); plt.show()

---
## 6. Complete Patch Embedding Module

Putting it all together into a reusable `nn.Module`:

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image to sequence of patch embeddings with [CLS] token and positional encoding."""

    def __init__(self, img_size=32, patch_size=4, in_channels=3, d_model=128):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        # Conv2d trick for patch projection
        self.patch_proj = nn.Conv2d(in_channels, d_model,
                                     kernel_size=patch_size, stride=patch_size)

        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # Learnable positional encoding
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model) * 0.02)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B = x.shape[0]

        # Patch projection: (B, C, H, W) → (B, d_model, grid_h, grid_w) → (B, N, d_model)
        x = self.patch_proj(x).flatten(2).transpose(1, 2)

        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, N+1, d_model)

        # Add positional encoding
        x = x + self.pos_embed

        return self.dropout(x)


# Test
patch_embed = PatchEmbedding(img_size=32, patch_size=4, in_channels=3, d_model=128)
images = torch.randn(8, 3, 32, 32)

with torch.no_grad():
    tokens = patch_embed(images)

print(f"Input images: {images.shape}")
print(f"Output tokens: {tokens.shape}")
print(f"  → {tokens.shape[1]} tokens = 1 [CLS] + {patch_embed.num_patches} patches")
print(f"  → Each token is {tokens.shape[2]}-dimensional")
print(f"\nTotal params: {sum(p.numel() for p in patch_embed.parameters()):,}")

---
## 7. Patch Size Trade-offs

Smaller patches = more tokens = more compute but potentially more detail:

In [ ]:
# Compare different patch sizes for CIFAR-10 (32×32)
img_size = 32
print(f"Image size: {img_size}×{img_size}\n")
print(f"{'Patch':>6} | {'Grid':>6} | {'Tokens':>7} | {'Patch dim':>10} | {'Note':>20}")
print("-" * 65)

for ps in [2, 4, 8, 16, 32]:
    if img_size % ps != 0:
        continue
    grid = img_size // ps
    n_tokens = grid * grid + 1  # +1 for CLS
    patch_dim = 3 * ps * ps
    note = ""
    if ps == 4:
        note = "← ViT default for CIFAR"
    elif ps == 16:
        note = "← ViT default for ImageNet"
    print(f"{ps:>4}×{ps} | {grid:>2}×{grid:<2} | {n_tokens:>7} | {patch_dim:>10} | {note:>20}")

print("\n⚠️ More tokens → quadratic cost in attention (O(N²))")
print("⚠️ Fewer tokens → each patch has less detail")

---
## 🧪 Exercises

### Exercise 1: Reconstruct from patches
Take the patch embeddings (before projection) and reconstruct the original image. Verify they're the same.

### Exercise 2: Overlapping patches
Implement patch embedding with `stride < patch_size` (overlapping patches). How many more tokens do you get?

### Exercise 3: Sinusoidal 2D positional encoding
Implement a 2D sinusoidal positional encoding that encodes row and column positions separately.

### Exercise 4: Patch size experiment
For CIFAR-10 (32×32), try patch sizes 2, 4, 8, and 16. Build a simple model and compare accuracy.

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Patch splitting** | Divide image into non-overlapping P×P patches |
| **Linear projection** | Flatten patch → project to d_model dimensions |
| **Conv2d trick** | `Conv2d(C, d, kernel=P, stride=P)` = patch embed |
| **[CLS] token** | Special learnable token for classification |
| **Positional encoding** | Learned or sinusoidal, added to patch embeddings |
| **Patch size trade-off** | Smaller patches = more tokens = more compute |

**Tomorrow:** We assemble everything into a complete Vision Transformer (ViT) and train it on CIFAR-10!